# Corporate Sales Data Cleaning & Analytics Pipeline
### Stage 1: Environment Setup & Data Ingestion

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load multi-sheet transactional data
sheets = pd.read_excel("../data/raw/practice dataset raw.xlsx", sheet_name=["Customers", "Transactions", "Orders"])
customers = sheets["Customers"]
transactions = sheets["Transactions"]
orders = sheets["Orders"]

### Stage 2: Deduplication and Imputation

In [ ]:
customers_cleaned = customers.drop_duplicates(subset=['customer_id'], keep='first').copy()
transactions_cleaned = transactions.drop_duplicates(subset=['transaction_id'], keep='first').copy()
orders_cleaned = orders.drop_duplicates(subset=['order_id'], keep='first').copy()

customers_cleaned['email'] = customers_cleaned['email'].fillna('Not Provided')
customers_cleaned['phone_primary'] = customers_cleaned['phone_primary'].fillna('Not Provided')
customers_cleaned['city'] = customers_cleaned['city'].fillna('Unknown')

country_map = {
    'United States': 'USA', 'usa': 'USA', 'US': 'USA', 'U.S.A.': 'USA', 'UNITED STATES': 'USA',
    'United Kingdom': 'UK', 'Great Britain': 'UK'
}
customers_cleaned['country'] = customers_cleaned['country'].replace(country_map)
orders_cleaned = orders_cleaned.drop(columns=['email'], errors='ignore')

### Stage 3: Feature Engineering & Logistics Audit

In [ ]:
customers_orders = pd.merge(customers_cleaned, orders_cleaned, on='customer_id', how='left')
full_data = pd.merge(customers_orders, transactions_cleaned, on='customer_id', how='left')

full_data['ship_days'] = (full_data['ship_date'] - full_data['order_date']).dt.days
valid_shipments = full_data[full_data['ship_days'] >= 0].copy()